# RG401

In [1]:
options(warn=-1)

In [2]:
library_load <- suppressMessages(
    
    suppressWarnings(
        
        list(

            library(stats), 
            library(emmeans), 
            library(outliers), # Grubbs outlier detection 
            
            # Data 
            library(tidyverse), 
            library(data.table), 
            library(reactable), 

            # Plotting 
            library(ComplexHeatmap), 
            library(patchwork), 
            library(cowplot), 
            library(ggrepel)

        )
    )
)

In [3]:
random_seed <- 42
set.seed(random_seed)

In [4]:
# Set working directory to project root
repo_root <- system("git rev-parse --show-toplevel", intern = TRUE)
setwd(repo_root)

In [5]:
# Plotting Theme
source("plotting_global.R")
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

# Parameter 

In [6]:
color$sample_group<- c("D0 +/+"="#66c2a5", "D0 cre/+"="#00634A", "D1 +/+"="#cd34b5", "D1 cre/+"="#FFAC1E", "D3 +/+"="#cd34b5", "D3 cre/+"="#FFAC1E", "D6 +/+"="#cd34b5", "D6 cre/+"="#FFAC1E")

# Function 

In [7]:
iqr <- function(data) {

    data <- data %>%
    
        group_by(sample_group, measurement) %>%
        mutate(
            
            Q1=quantile(value, 0.25, na.rm=TRUE),
            Q3=quantile(value, 0.75, na.rm=TRUE),
            IQR=Q3 - Q1,
            outlier=value < (Q1 - 1.5*IQR) | value > (Q3 + 1.5*IQR)
        
        ) %>% dplyr::filter(!outlier)

    return(data)
}

In [8]:
data_stat <- function(data, assay="FACS") {

    if(assay=="FACS") {
        
        data <- data %>% dplyr::group_by(measurement, dpi, genotype, sample_group, tissue)
    
    } else {

        data <- data %>% dplyr::group_by(measurement, dpi, genotype, sample_group)
        
    }
    
    data <- data %>% 

        dplyr::summarise(
            
            value_mean=mean(value), 
            value_sd=sd(value), 
            n=n(), 
            value_se=value_sd / sqrt(n),
            value_se_min=value_mean-value_se, 
            value_se_max=value_mean+value_se, 
            .groups="drop"
        
        ) 

    return(data)
    
}

In [9]:
pl <- function(data, stat, assay="") {
    
    p <- lapply(split(stat, f=stat$measurement), function(x) {
        
        y_limit <- max(c(abs(x$value_se_min), abs(x$value_se_max)))
        
        p_i <- ggplot(x, aes(x=dpi, y=value_mean, fill=sample_group, group=genotype)) +
            
            ggtitle(x$measurement[1]) +

            
        
            geom_bar(
                stat="identity", color="black", linewidth=0.1, width=0.8, position=position_dodge(width=0.8)
            ) +

            geom_errorbar(
                aes(ymin=value_se_min, ymax=value_se_max),
                width=0.4,
                colour="black",
                linewidth=0.1,
                position=position_dodge(width=0.8)
            ) +

        geom_hline(yintercept=0, size=0.5) + 
    
            geom_point(
                data=data[data$measurement == x$measurement[1], ],
                aes(x=dpi, y=value, fill=sample_group, group=genotype),
                position=position_jitterdodge(jitter.width=0.12, jitter.height=0, dodge.width=0.8, seed=1), 
                size=1.0, stroke=0.1, alpha=1, shape=21, colour="black", inherit.aes=FALSE
            ) +

            # scale_y_continuous(limits=c(-y_limit, +y_limit)) + 
            scale_fill_manual(values=color$sample_group) + 
            facet_grid(~dpi, scales="free") +
            theme(legend.position="none") +
            theme_global_set(4)
        
        if(assay=="FACS") {p_i <- p_i + facet_grid(~tissue+dpi, scales="free")}

        return(p_i)
        
    }
          )
    
    p <- lapply(p, function(p) egg::set_panel_size(p, width=unit(0.5, "cm"), height=unit(2.0, "cm")))
    p <- do.call(gridExtra::arrangeGrob, c(p, ncol=4, nrow=ceiling(length(p)/4)))
    
    return(p)

}

In [10]:
anova <- function(data) {

    data$time <- factor(data$dpi, levels = c("Ctl","D1","D3","D6")) %>% droplevels()
    data$genotype <- factor(data$genotype, levels = c("+/+","cre/+")) %>% droplevels()

    data <- split(data, f=data$measurement)

    model <- lapply(data, function(x) {stats::aov(value ~ genotype * dpi, data=x)})
    emm <- lapply(model, function(x) {emmeans(x, ~ genotype * dpi)})

    dpi_res <- lapply(emm, function(x) {contrast(x, method = "pairwise", by = "dpi", adjust = "sidak") %>% data.frame() %>% dplyr::filter(p.value <= 0.05)})
    genotype_res <- lapply(emm, function(x) {contrast(x, method = "pairwise", by = "genotype", adjust = "sidak") %>% data.frame() %>% dplyr::filter(p.value <= 0.05)})

    return(list(dpi_res, genotype_res))
    

    
}

# Spleen RG357

In [11]:
data <- openxlsx::read.xlsx("data/validation/RG357/qPCR/qpcr_facs.xlsx") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [12]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value") %>% dplyr::arrange(measurement)

In [13]:
# data <- iqr(data)
stat <- data_stat(data, assay="FACS")
p <- pl(data, stat, assay="FACS")

In [14]:
pdf("result/validation/qpcr_facs_RG357.pdf", width=20, height=2.0*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

agg_record_160823610 
                   2

# Spleen RG401

In [15]:
data <- openxlsx::read.xlsx("data/validation/RG401/qPCR/qpcr_spleen.xlsx") %>% dplyr::mutate(genotype=gsub("'", "", genotype), dpi=ifelse(dpi=="Ctl", "D0", dpi), sample_group=paste(dpi, genotype))

In [16]:
# data <- data %>% dplyr::filter(dpi!="D6") %>% droplevels()

In [17]:
data <- data %>% pivot_longer(cols=-c(tissue, genotype, treatment, dpi, sample_group), names_to="measurement", values_to="value") %>% dplyr::arrange(measurement) %>% dplyr::mutate(value=-log2(value))

In [18]:
data <- iqr(data)
stat <- data_stat(data)
p <- pl(data, stat)

In [19]:
pdf("result/validation/qpcr_spleen_RG401.pdf", width=7.5, height=2.0*ceiling(length(p)/5))

grid::grid.draw(p)

dev.off()

agg_record_1114738950 
                    2